# Inspect the results of the assessment of a classifier's fairness based on movement patterns, focusing on single-cell candidates

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import pickle
from pathlib import Path

import folium

In [ ]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

For each candidate, retrieve the grid and subset of cell it refers to.

In [ ]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Retrieve the flattened list of object IDs associated with the candidates.
flattened_list_candidates = dict_candidates['flat_ids']

# Compute the starting pos, ending pos, and number of objects of each candidate.
start_pos_candidates = dict_candidates['start_pos'][:-1]
end_pos_candidates = dict_candidates['start_pos'][1:]
num_candidates = start_pos_candidates.size
num_objs_candidates = np.diff(dict_candidates['start_pos'])

In [ ]:
# Retrieve the results computed for a given group of datasets.
# display(vec_lr_dataset)


# For each candidate, here represented as a tuple of cell IDs, associate the grid and subset of cells it refers to.
list_grid_ids = np.empty(num_candidates, dtype=object)
list_cellids = np.empty(num_candidates, dtype=object)
count = 0
for grid in grid_info:
    # display(grid)
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=object)
    grid_id[0] = (int(grid[1]), int(grid[2]))
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    list_grid_ids[count : count + num_els_grid] = grid_id
    list_cellids[count : count + num_els_grid] = cell_ids

    count += num_els_grid


# Put all the information in a pandas Dataframe.
df_candidates = pd.DataFrame({"grid_id":   list_grid_ids,
                              "cell_ids":  list_cellids})
#
# Count the number of cells making up each candidate.
# NOTE: We use numpy's 'fromiter' because it's way faster (C-backed code) than using pandas' apply/map + lamba func on a series.
cell_ids = df_candidates['cell_ids'].to_numpy()
df_candidates['num_cells'] = np.fromiter(
    (len(x) if type(x) is tuple else 1 for x in cell_ids),
    dtype=np.uint32,
    count=cell_ids.size
)
df_candidates.index.name = 'candidate_id'
display(df_candidates)


# Free some memory.
del cell_ids, list_grid_ids, list_cellids

In [ ]:
# Find out the list of files of a group of unfair datasets, and the results obtained from them.
name_set_groups_datasets = 'num_hotspots'
path_unfair_datasets = Path(f'./experiments/{name_set_groups_datasets}/')
list_files_results = sorted([f for f in path_unfair_datasets.iterdir() if f.is_file() and 'results' in f.name])
list_files_datasets = sorted([f for f in path_unfair_datasets.iterdir() if f.is_file() and 'results' not in f.name])
display(list_files_results)


# Now read the data of a specific group of dataset and the related assessment.
idx_selected_group = 2
path_group_datasets = list_files_datasets[idx_selected_group]
path_group_results = list_files_results[idx_selected_group]
# display(path_group_results)
with open(path_group_datasets, "rb") as f: set_datasets = pickle.load(f)
with open(path_group_results, "rb") as f: set_results = pickle.load(f)

In [ ]:
idx_dataset = 747

# Retrieve the multipolygons of the hotspots generated for the considered dataset.
multipoly_hotspots = set_datasets['data'][idx_dataset][0]

# Retrieve the IDs, log-LRs, and cells' polygons of the extreme candidates found for this dataset.
df_extreme_candidates = pd.DataFrame(data = set_results['lr_candidates'][idx_dataset], 
                                     index = set_results['idx_candidates'][idx_dataset],
                                     columns=['lr_candidates'])
df_extreme_candidates[['grid_id', 'cell_ids']] = \
    df_candidates.loc[df_extreme_candidates.index, ['grid_id', 'cell_ids']]
df_extreme_candidates['grid_res'] = df_extreme_candidates['grid_id'].map(lambda x : x[0])


# Build the multipolygons associated with each extreme candidate.
list_mp_candidates = []
for grid_id, cell_ids in zip(df_extreme_candidates['grid_id'], df_extreme_candidates['cell_ids']) :
    list_mp_candidates.append( dict_grids[grid_id].grid.loc[np.atleast_1d(cell_ids)].union_all() )
df_extreme_candidates['geometry'] = list_mp_candidates
df_extreme_candidates = gpd.GeoDataFrame(df_extreme_candidates, geometry='geometry', crs=dict_grids[grid_id].grid.crs)

# display(df_candidates)
display(df_extreme_candidates)

For a given dataset in a given group, we now plot the hotspots of unfairness and also the candidate subsets of cells that have been deemed as extreme. The intention is to see how well our approach can visually pinpoint the hotspots at the different grid resolutions. 

In [ ]:
import folium

# Plot the multipolygon of the hotspots of a selected dataset.

# Base map centered on candidates
minx, miny, maxx, maxy = multipoly_hotspots[0].bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=12, prefer_canvas=True)


# Plot the hotspots of the considered dataset.
color_hotspots = ['blue', 'red', 'green', 'black']
iter_colors = iter(color_hotspots)
for mp in multipoly_hotspots :
    color = next(iter_colors)
    folium.GeoJson(mp, style_function = lambda feature, color=color : {"weight": 0,
                                                                       "fillColor": color,
                                                                       "fillOpacity": 0.5},).add_to(m)


# Plot the bounding box in which the grids were materialized. We do this to understand the area in which the objects' stops
# are located.
plot_bbox = False
if plot_bbox :
    any_grid = next(iter(dict_grids.values())).grid.union_all()
    folium.GeoJson(any_grid, style_function = lambda feature:{"color": "black",
                                                              "weight": 1,
                                                              "fillOpacity": 0}).add_to(m)


# Now plot the extreme candidates.
plot_candidates = False
if plot_candidates :    
    single_candidate = False
    # Case in which we want to plot a single extreme candidate.
    if single_candidate :
        idx_candidate = 4630471
        grid_candidate = dict_grids[df_extreme_candidates.loc[idx_candidate, 'grid_id']].grid
        def style_fn1(feature):
            return {"color": "blue",
                    "weight": 0.1,
                    "fillColor": "yellow",
                    "fillOpacity": 0.5,}
        folium.GeoJson(grid_candidate, style_function=style_fn1).add_to(m)
    
        def style_fn2(feature):
            return {"color": "blue",
                    "weight": 0.5,
                    "fillColor": "blue",
                    "fillOpacity": 0,}
        folium.GeoJson(df_extreme_candidates.loc[idx_candidate, 'geometry'], style_function=style_fn2).add_to(m)

    # Case in which we want to plot a sets of candidates from various grid resolutions.
    else :
        # Here we select the resolutions of the candidates we want to plot.
        # If the list is empty, we consider all the candidates.
        list_resolutions = [200]
        sel_extreme_candidates = df_extreme_candidates.loc[df_extreme_candidates['grid_res'].isin(list_resolutions)] if list_resolutions else df_extreme_candidates
        def style_fn3(feature):
            return {"color": "yellow",
                    "weight": 0.5,
                    "fillColor": "yellow",
                    "fillOpacity": 0.5,}
        folium.GeoJson(sel_extreme_candidates, style_function=style_fn3).add_to(m)


m